# Phase 1: SFT on Colab using Unsloth 🚀

**Optimized for free Colab T4 GPU (15GB VRAM)**

Key optimizations:
- Pre-quantized 4-bit model (2x faster loading)
- Sequence packing (~30% speedup)
- Gradient checkpointing (70% less memory)
- Checkpoint saving (survives Colab disconnects)

In [ ]:
%%capture
# Install Unsloth + all dependencies (clean single install)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl" peft accelerate bitsandbytes huggingface_hub
# Upgrade datasets separately — Colab default version is too old (missing Json feature type)
!pip install -q -U datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from datasets import load_dataset

# 1. Load data from HuggingFace
dataset = load_dataset("myounes21/logos-reasoning-dataset", split="train")

# Qwen Chat Template formatting
chat_template = "<|im_start|>user\n{instruction}<|im_end|>\n<|im_start|>assistant\n<think>\n{think}\n</think>\n```python\n{answer}\n```<|im_end|>"

def format_prompt(example):
    prompt = chat_template.format(
        instruction=example['instruction'],
        think=example['think'],
        answer=example['answer']
    )
    return {"text": prompt}

formatted_dataset = dataset.map(format_prompt)
split_dataset = formatted_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset['train']
eval_dataset = split_dataset['test']
print(f"✅ Loaded {len(train_dataset)} training examples and {len(eval_dataset)} validation examples!")

In [ ]:
# Sanity check — verify data format before training
sample = train_dataset[0]
print("Fields:", list(sample.keys()))
print("---")
print(sample['text'][:800])
print("---")
print(f"Sample text length: {len(sample['text'])} chars")

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
import torch

max_seq_length = 2048  # T4-safe (down from 4096)
dtype = None  # Auto-detect: float16 for T4, bfloat16 for Ampere+
load_in_4bit = True

# 2. Load pre-quantized model (2x faster than quantizing on the fly)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 3. Apply LoRA (Unsloth optimized)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,  # Unsloth specifically optimizes 0 dropout
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # 70% less memory
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

print(f"✅ Model loaded! Trainable params: {model.print_trainable_parameters()}")
!nvidia-smi

## 🧪 Smoke Test (Run this first to verify everything works)

In [ ]:
from trl import SFTTrainer, SFTConfig

# Clear any leftover VRAM
import gc
gc.collect()
torch.cuda.empty_cache()

# SMOKE TEST — 60 steps only
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        packing = True,  # ~30% speedup — packs short samples together
        dataset_num_proc = 2,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,  # Effective batch size = 4
        max_steps = 60,
        learning_rate = 2e-4,
        lr_scheduler_type = "cosine",
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        seed = 3407,
        output_dir = "/content/drive/MyDrive/logos-sft-smoke-test",
        report_to = "none",
    ),
)

# Unsloth loss masking — only train on assistant responses
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

print("🚀 Starting smoke test (60 steps)...")
trainer.train()
print("✅ Smoke test passed! VRAM usage:")
!nvidia-smi

## 🏋️ Full Training Run (3 Epochs)

**⚠️ Only run this after the smoke test passes!**

Checkpoints are saved every 50 steps to Google Drive, so you won't lose progress if Colab disconnects.

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers.trainer_utils import get_last_checkpoint
import torch
import gc

# Clear VRAM
gc.collect()
torch.cuda.empty_cache()

OUTPUT_DIR = "/content/drive/MyDrive/logos-sft-unsloth-full"

# Create trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        packing=True,
        dataset_num_proc=2,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=7,
        num_train_epochs=3,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="steps",
        save_steps=25,
        save_total_limit=3,
        load_best_model_at_end=False,
        optim="adamw_8bit",
        weight_decay=0.01,
        seed=3407,
        output_dir=OUTPUT_DIR,
        report_to="none",
    ),
)

# Train only on assistant responses
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user
",
    response_part="<|im_start|>assistant
",
)

# Resume automatically if checkpoint exists
checkpoint = get_last_checkpoint(OUTPUT_DIR)

print("🏋️ Starting training...")

if checkpoint is not None:
    print(f"🔄 Resuming from checkpoint: {checkpoint}")
    trainer.train(resume_from_checkpoint=checkpoint)
else:
    print("🆕 No checkpoint found. Starting fresh training.")
    trainer.train()

# Save final adapter
ADAPTER_DIR = "/content/drive/MyDrive/logos-sft-adapter-unsloth-final"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"
✅ Training complete!")
print(f"📁 Final adapter saved to: {ADAPTER_DIR}")


## 📤 Push to HuggingFace Hub (Optional)

Run this to upload your trained adapter to HuggingFace.
You need to be logged in: run `huggingface-cli login` or set `HF_TOKEN`.

In [ ]:
# Optional: Login to HuggingFace
from huggingface_hub import login
login()  # This will prompt for your token

# Push adapter to Hub
HF_REPO = "myounes21/logos-sft-adapter"  # Change this to your repo name
model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)
print(f"✅ Adapter pushed to: https://huggingface.co/{HF_REPO}")